"""
You are a Senior Data Engineer at a ride-sharing company.

You receive ~500GB of trip data daily in Parquet format.

Schema:

- trip_id (string)
- driver_id (string)
- city (string)
- fare_amount (double)
- trip_distance (double)
- trip_timestamp (timestamp)

Business Requirements:

1. Identify the Top 3 drivers per city per day based on total fare.
2. Data may arrive late and may contain duplicate records.
3. Some cities generate 10x more data than others (data skew).
4. Output must support fast downstream analytical queries.
5. The solution must scale reliably as data grows.

How would you implement this in PySpark?
"""

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType,LongType
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# spark = SparkSession.builder.appName('TransactionCleanup').getOrCreate()

schema = StructType([
    StructField('trip_id', StringType(), True),
    StructField('driver_id', StringType(), True),
    StructField('city', StringType(), True),
    StructField('fare_amount', IntegerType(), True),
    StructField('trip_distance',IntegerType(), True),
    StructField('trip_timestamp', LongType(), True)

])


total_records = 1000
num_partitions = 10
cities = [
    
]
df = spark.range(10, total_records,numPartitions=num_partitions)

from pyspark.sql import functions as F

fake_rides_df = df.select(
    
    F.expr("uuid()").alias("trip_id"),
    
    F.element_at(
        F.array(
            F.lit("driver_1"), F.lit("driver_2"), F.lit("driver_3"), F.lit("driver_4"),
            F.lit("driver_5"), F.lit("driver_6"), F.lit("driver_7"), F.lit("driver_8"),
            F.lit("driver_9"), F.lit("driver_10"), F.lit("driver_11"), F.lit("driver_12"),
            F.lit("driver_13"), F.lit("driver_14"), F.lit("driver_15"), F.lit("driver_16"),
            F.lit("driver_17"), F.lit("driver_18"), F.lit("driver_19"), F.lit("driver_20")
        ),
        (F.floor(F.rand()*20)+1).cast("int")
    ).alias("driver_id"),
    
    F.element_at(
        F.array(
            F.lit("New York"), F.lit("London"), F.lit("Paris"), F.lit("Tokyo"),
            F.lit("Dubai"), F.lit("Singapore"), F.lit("Sydney"), F.lit("Toronto"),
            F.lit("Berlin"), F.lit("San Francisco"), F.lit("Chicago"),
            F.lit("Los Angeles"), F.lit("Mumbai"), F.lit("Delhi"),
            F.lit("Bangalore"), F.lit("Hyderabad"), F.lit("Chennai"),
            F.lit("Kolkata"), F.lit("Pune"), F.lit("Amsterdam")
        ),
        (F.floor(F.rand()*20)+1).cast("int")
    ).alias("city"),
    
    F.round(F.rand()*100, 2).alias("fare_amount"),
    
    F.round(F.rand()*10, 2).alias("trip_distance"),
    
    F.expr("unix_timestamp(current_timestamp()) - int(rand()*604800)").alias("trip_timestamp")
)

fake_rides_df.show()

In [0]:
rides_df = fake_rides_df.withColumn('ride_date', F.to_date(F.from_unixtime(F.col('trip_timestamp').cast("long"))))
display(rides_df)

Identify the Top 3 drivers per city per day based on total fare.****

In [0]:
from pyspark.sql.window import Window

grouped_df = rides_df.groupBy(F.col('ride_date'),F.col('driver_id'))\
    .agg(
        F.round(F.sum(F.col('fare_amount')),2).alias('total_fare')
        )

window_spec = Window.partitionBy(F.col('ride_date')).orderBy(F.col('total_fare'))

ranked_df = grouped_df.withColumn('rank', F.rank().over(window_spec))
ranked_df.filter(F.col('rank') <= 3).show(truncate=False)

Handling Data Skew (The Salting Technique)

In [0]:
SALT_FACTOR = 20

df_salted = rides_df.withColumn("salt", F.floor(F.rand() * SALT_FACTOR))
# df_salted.show()

from pyspark.sql.window import Window

grouped_df = rides_df.groupBy(F.col('ride_date'),F.col('driver_id'))\
    .agg(
        F.round(F.sum(F.col('fare_amount')),2).alias('total_fare')
        )

window_spec = Window.partitionBy(F.col('ride_date')).orderBy(F.col('total_fare'))

ranked_df = grouped_df.withColumn('rank', F.row_number().over(window_spec))
ranked_df.filter(F.col('rank') <= 3).show(truncate=False)

Handling Late Data & Duplicates (Idempotency)

In [0]:
df_dedup = rides_df.dropDuplicates(["trip_id"])
from delta.tables import DeltaTable

def upsert_to_delta(micro_batch_df, batch_id):
    delta_table = DeltaTable.forPath(spark, "hdfs://gold/driver_rankings")
    
    delta_table.alias("target").merge(
        micro_batch_df.alias("source"),
        "target.driver_id = source.driver_id AND target.city = source.city AND target.trip_date = source.trip_date"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

| Strategy      | Implementation                                 | Benefit                                                                                  |
|---------------|------------------------------------------------|------------------------------------------------------------------------------------------|
| AQE           | spark.sql.adaptive.enabled = true              | Automatically handles skew and optimizes joins at runtime.                               |
| Partitioning  | .partitionBy("trip_date", "city")              | Allows downstream queries to "prune" partitions, scanning only relevant data.            |
| Z-Order       | OPTIMIZE table ZORDER BY (driver_id)           | Colocates related data within files for faster lookup in Delta Lake.                     |
| File Sizing   | maxRecordsPerFile = 1000000                    | Prevents the "Small File Problem" which slows down S3 reads.                             |